# Stage 6E — Bounded Strategy–Result Interpretation and Synthesis

This notebook converts the validated Stage 6D linkage screening into bounded, evidence-class-aware interpretations.

The synthesis keeps direct descriptive linkage, temporal alignment, company-reported attribution, reporting-perimeter context, and evidence boundaries separate. Track A portfolio-performance findings remain authoritative for breadth, selected-category leadership, longitudinal stability/momentum, persistence, ownership sensitivity, and overall-winner defensibility.

Stage 6E creates no causal-effect estimate, strategy-effect ranking, composite score, weighting framework, visualization, report, README, or revised overall winner.


## Environment Setup

Lock the authoritative Stage 6D commit and define the exact governed evidence used for Stage 6E.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "c9ccd052bd0b0946100d45f2ef448806adabc497"

INPUT_LOCKS = {
    "data/analytical/stage6d_action_linkage_screening.csv":
        "a93aa4d2bc61f88af824d1179398e06671638a70d8ba219b0202ef441dc06a46",
    "data/analytical/stage6d_company_claim_screening.csv":
        "a30a1a67ed3dcc1e41fdefb2b73f5f2e40b34ec54c3a4ae5a59d1a8af9189b5b",
    "data/analytical/stage6d_portfolio_context_screening.csv":
        "291c6faf197479a3c1680f9bb43bede684022a48a17b5bcd503e65c6b04bf478",
    "data/analytical/stage6d_screening_summary.csv":
        "0899104f54e8b8e6b49720a5161b7152b14d88f1c7b748a2b1a12f3984a34395",
    "metadata/stage6d_screening_exceptions.csv":
        "58d5ad923c49d49f305a6d509dc5ed82066a96d8fccca7cc08171e9ff630356b",
    "metadata/stage6d_linkage_validation.csv":
        "265f9caf51859de30775d77a280dde1c5f2bdaf03edf9e01d1f698681e6c806b",
    "data/analytical/stage6c_strategy_actions.csv":
        "092a12a92f982a93101ea31e049f1c2292cedf54aa7decd157952bec185f3822",
    "data/analytical/stage6c_result_observations.csv":
        "bc91bd300a01d3eecbc7c7b81d60747f203a4a63aff39f890aaffafc67ab9ad1",
    "data/analytical/stage6c_company_attribution_claims.csv":
        "44aaf59c2bd454b486b48ca8eb0f762cc19124f55cafea408bb7a44460b6f2e1",
    "data/analytical/stage4_final_findings.csv":
        "847d17369b60f950495467a9ecedbbbd2bd5397d49aa6a845be6dd784b8ee35c",
    "metadata/stage5_research_questions.csv":
        "6f2854d93713534e89f64ff743d4ec2a0bdd101e9a109989cb13bfcecd1aaa14",
}

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE6E_OUTPUT_ROOT", "/content/fmcg_stage6e_outputs")
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)

print(f"Locked Stage 6D commit: {INPUT_COMMIT}")
print(f"Required governed inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")


Locked Stage 6D commit: c9ccd052bd0b0946100d45f2ef448806adabc497
Required governed inputs: 11
Output root: /content/fmcg_stage6e_outputs


## Locked Input Retrieval

Retrieve only the 11 governed inputs from the authoritative Stage 6D commit. Colab uses `GITHUB_TOKEN`; local validation may use `FMCG_STAGE6E_INPUT_ROOT`.


In [2]:
configured_root = os.environ.get("FMCG_STAGE6E_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run in Google Colab or set FMCG_STAGE6E_INPUT_ROOT."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Colab Secret GITHUB_TOKEN is unavailable.")

    INPUT_ROOT = Path("/content/fmcg_stage6e_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}"
            f"/contents/{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage6e-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"Input retrieval failed for {relative_path}: HTTP {exc.code}"
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing = [p for p in INPUT_LOCKS if not (INPUT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing Stage 6E inputs: {missing}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 11/11


## Input Integrity and Evidence Registries

Verify inherited SHA-256 values, confirm the Stage 6D gate, and load the action, result, claim, Track A, and research-question registries.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

lock_rows = []
for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append({
        "file_path": relative_path,
        "expected_sha256": expected_sha256,
        "actual_sha256": actual_sha256,
        "hash_match": actual_sha256 == expected_sha256,
        "locked_repository_commit": INPUT_COMMIT,
    })

stage6e_input_lock = pd.DataFrame(lock_rows)

if not stage6e_input_lock["hash_match"].all():
    failed = stage6e_input_lock.loc[
        ~stage6e_input_lock["hash_match"], "file_path"
    ].tolist()
    raise RuntimeError(f"Stage 6E input checksum failure: {failed}")

linkages = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6d_action_linkage_screening.csv",
    dtype=str, keep_default_na=False,
)
claim_screening = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6d_company_claim_screening.csv",
    dtype=str, keep_default_na=False,
)
portfolio_context = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6d_portfolio_context_screening.csv",
    dtype=str, keep_default_na=False,
)
screening_summary = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6d_screening_summary.csv",
    dtype=str, keep_default_na=False,
)
stage6d_exceptions = pd.read_csv(
    INPUT_ROOT / "metadata/stage6d_screening_exceptions.csv",
    dtype=str, keep_default_na=False,
)
stage6d_validation = pd.read_csv(
    INPUT_ROOT / "metadata/stage6d_linkage_validation.csv",
    dtype=str, keep_default_na=False,
)
actions = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_strategy_actions.csv",
    dtype=str, keep_default_na=False,
)
results = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_result_observations.csv",
    dtype=str, keep_default_na=False,
)
company_claims = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_company_attribution_claims.csv",
    dtype=str, keep_default_na=False,
)
stage4_findings = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage4_final_findings.csv",
    dtype=str, keep_default_na=False,
)
research_questions = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_research_questions.csv",
    dtype=str, keep_default_na=False,
)

final_stage6d = stage6d_validation.loc[
    stage6d_validation["check_id"] == "S6D040"
].iloc[0]

if (
    final_stage6d["result"] != "PASS_WITH_CAVEAT"
    or final_stage6d["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6D final gate.")

expected_cardinalities = {
    "linkages": (len(linkages), 22),
    "claim_screening": (len(claim_screening), 6),
    "portfolio_context": (len(portfolio_context), 4),
    "screening_summary": (len(screening_summary), 5),
    "stage6d_exceptions": (len(stage6d_exceptions), 12),
    "stage6d_validation": (len(stage6d_validation), 40),
    "actions": (len(actions), 22),
    "results": (len(results), 69),
    "company_claims": (len(company_claims), 6),
    "stage4_findings": (len(stage4_findings), 6),
    "research_questions": (len(research_questions), 11),
}

for label, (actual, expected) in expected_cardinalities.items():
    if actual != expected:
        raise RuntimeError(f"{label}: expected {expected}, found {actual}")

action_lookup = actions.set_index("action_id").to_dict("index")
result_lookup = results.set_index("result_id").to_dict("index")
claim_lookup = company_claims.set_index("claim_id").to_dict("index")
finding_lookup = stage4_findings.set_index("finding_id").to_dict("index")

valid_action_ids = set(action_lookup)
valid_result_ids = set(result_lookup)
valid_claim_ids = set(claim_lookup)
valid_finding_ids = set(finding_lookup)

def summarize_results(result_ids: str) -> str:
    ids = [x for x in result_ids.split(";") if x]
    parts = []
    for result_id in ids:
        row = result_lookup[result_id]
        parts.append(
            f"{result_id}: {row['metric_name']}={row['value_numeric']} "
            f"{row['unit']} ({row['reference_period']})"
        )
    return " | ".join(parts)

print(f"Input checksums passed: {len(stage6e_input_lock)}/{len(stage6e_input_lock)}")
print(f"Stage 6D gate: {final_stage6d['result']} / {final_stage6d['status']}")
print(
    f"Actions: {len(actions)} | Results: {len(results)} | "
    f"Claims: {len(company_claims)} | Track A findings: {len(stage4_findings)}"
)


Input checksums passed: 11/11
Stage 6D gate: PASS_WITH_CAVEAT / passed_with_caveat
Actions: 22 | Results: 69 | Claims: 6 | Track A findings: 6


## Bounded Case Interpretations

Represent all 22 documented actions and all six company-attribution claims in case-level interpretations without upgrading evidence class.


In [4]:
case_specs = [
    {
        "case_id":"S6ECASE01","canonical_group":"Wings Group",
        "case_title":"Product launches without compatible result observations",
        "strategy_ids":"STR01",
        "action_ids":"S6CACT_WNG_001;S6CACT_WNG_004;S6CACT_WNG_006;S6CACT_WNG_008",
        "result_ids":"","company_claim_ids":"","stage4_context_ids":"FND4_02;FND4_03",
        "interpretation_class":"evidence_boundary","evidence_status":"not_assessable",
        "bounded_interpretation":"Multiple Wings product launches are documented, but no compatible brand/category result can be linked to them.",
        "allowed_public_claim":"Wings had documented product-launch activity during the eligible period.",
        "prohibited_inference":"Do not attribute category leadership, stability, sales, or profitability to these launches.",
        "critical_caveat":"Track A leadership/stability remains group-level context."
    },
    {
        "case_id":"S6ECASE02","canonical_group":"Wings Group",
        "case_title":"Pricing, distribution, and marketing actions without compatible outcomes",
        "strategy_ids":"STR02;STR03;STR04",
        "action_ids":"S6CACT_WNG_002;S6CACT_WNG_003;S6CACT_WNG_005;S6CACT_WNG_007",
        "result_ids":"","company_claim_ids":"","stage4_context_ids":"FND4_02;FND4_03",
        "interpretation_class":"evidence_boundary","evidence_status":"not_assessable",
        "bounded_interpretation":"Pack-price, channel-availability, and marketing actions are documented, but compatible commercial outcomes are unavailable.",
        "allowed_public_claim":"These actions may be described as documented commercial execution.",
        "prohibited_inference":"Do not infer pricing, marketing, distribution, consumer-reach, or financial effectiveness.",
        "critical_caveat":"Lower public result disclosure is an evidence limitation, not weak performance."
    },
    {
        "case_id":"S6ECASE03","canonical_group":"Indofood",
        "case_title":"Parent integrated-model explanations",
        "strategy_ids":"STR05","action_ids":"",
        "result_ids":"S6CRES_IDF_2024_OUT07;S6CRES_IDF_2024_OUT13;S6CRES_IDF_2025_OUT07;S6CRES_IDF_2025_OUT13",
        "company_claim_ids":"S6CCLM_IDF_001;S6CCLM_IDF_002",
        "stage4_context_ids":"FND4_01;FND4_03;FND4_04",
        "interpretation_class":"company_reported_attribution","evidence_status":"company_reported",
        "bounded_interpretation":"Indofood reported sales/profit outcomes and attributed performance to its integrated model, but no matching discrete action was extracted.",
        "allowed_public_claim":"Management attributed FY2024 and FY2025 performance to the integrated model.",
        "prohibited_inference":"Do not present the integrated model as an independently identified cause of the reported results.",
        "critical_caveat":"Parent-consolidated results include businesses beyond primary packaged FMCG."
    },
    {
        "case_id":"S6ECASE04","canonical_group":"Indofood",
        "case_title":"ICBP FY2024 volume and productivity explanation",
        "strategy_ids":"STR08","action_ids":"",
        "result_ids":"S6CRES_ICBP_2024_OUT07;S6CRES_ICBP_2024_OUT13",
        "company_claim_ids":"S6CCLM_ICBP_001","stage4_context_ids":"FND4_03",
        "interpretation_class":"company_reported_attribution","evidence_status":"company_reported",
        "bounded_interpretation":"ICBP reported FY2024 sales/profit outcomes and management attributed improvement to higher volume and productivity/efficiency.",
        "allowed_public_claim":"The management explanation may be reported as company-reported.",
        "prohibited_inference":"Do not treat volume or productivity as independently established causes.",
        "critical_caveat":"ICBP consolidated outcomes include overseas operations."
    },
    {
        "case_id":"S6ECASE05","canonical_group":"Mayora",
        "case_title":"Product innovation and FY2025 consolidated sales",
        "strategy_ids":"STR01","action_ids":"S6CACT_MYR_001",
        "result_ids":"S6CRES_MYR_2025_OUT07;S6CRES_MYR_2024_OUT07",
        "company_claim_ids":"","stage4_context_ids":"FND4_05;FND4_06",
        "interpretation_class":"temporal_alignment","evidence_status":"temporally_aligned",
        "bounded_interpretation":"Product innovation is documented in 2025 and overlaps the FY2025 consolidated sales period, without an isolated brand-level effect.",
        "allowed_public_claim":"The action and company-level sales are temporally aligned.",
        "prohibited_inference":"Do not state that product innovation caused sales performance.",
        "critical_caveat":"Mayora results mix domestic and export/international activity."
    },
    {
        "case_id":"S6ECASE06","canonical_group":"Mayora",
        "case_title":"Pricing policy, price adjustment, and FY2025 sales",
        "strategy_ids":"STR02","action_ids":"S6CACT_MYR_002",
        "result_ids":"S6CRES_MYR_2025_OUT07;S6CRES_MYR_2024_OUT07",
        "company_claim_ids":"S6CCLM_MYR_001","stage4_context_ids":"FND4_05;FND4_06",
        "interpretation_class":"company_reported_attribution","evidence_status":"company_reported",
        "bounded_interpretation":"Competitive pricing is documented, while management separately reported that raw-material inflation required selling-price adjustments and affected sales-target achievement.",
        "allowed_public_claim":"The pricing/sales explanation may be reported as company-reported.",
        "prohibited_inference":"Do not collapse the policy and price-adjustment explanation into one independently verified causal chain.",
        "critical_caveat":"Consolidated sales mix domestic and export/international activity."
    },
    {
        "case_id":"S6ECASE07","canonical_group":"Mayora",
        "case_title":"Share buyback and consumer-market outcomes",
        "strategy_ids":"STR07","action_ids":"S6CACT_MYR_003",
        "result_ids":"","company_claim_ids":"","stage4_context_ids":"FND4_06",
        "interpretation_class":"evidence_boundary","evidence_status":"not_assessable",
        "bounded_interpretation":"The share buyback is documented but no compatible consumer or operational outcome is attributable to it.",
        "allowed_public_claim":"The buyback may be described as capital-allocation context.",
        "prohibited_inference":"Do not treat the buyback as evidence of consumer-market success or failure.",
        "critical_caveat":"Capital allocation and consumer-market performance are separate constructs."
    },
    {
        "case_id":"S6ECASE08","canonical_group":"Mayora",
        "case_title":"Domestic-input sourcing and FY2025 operating profit",
        "strategy_ids":"STR05","action_ids":"S6CACT_MYR_004",
        "result_ids":"S6CRES_MYR_2025_OUT13;S6CRES_MYR_2024_OUT13",
        "company_claim_ids":"S6CCLM_MYR_002","stage4_context_ids":"FND4_05;FND4_06",
        "interpretation_class":"temporal_alignment","evidence_status":"temporally_aligned",
        "bounded_interpretation":"Domestic-input sourcing overlaps FY2025 operating profit while raw-material cost pressure remains a material alternative explanation.",
        "allowed_public_claim":"The sourcing action and operating-profit result may be described as temporally aligned with caveats.",
        "prohibited_inference":"Do not state that domestic sourcing improved or protected operating profit.",
        "critical_caveat":"Mixed geography and input-cost pressure prevent isolated effect interpretation."
    },
    {
        "case_id":"S6ECASE09","canonical_group":"Unilever Indonesia",
        "case_title":"Company-level distribution and promotion with FY2024 sales",
        "strategy_ids":"STR03;STR04","action_ids":"S6CACT_UNV_001;S6CACT_UNV_002",
        "result_ids":"S6CRES_UNV_ORIG_2024_OUT07;S6CRES_UNV_ORIG_2023_OUT07",
        "company_claim_ids":"","stage4_context_ids":"FND4_02;FND4_03;FND4_04",
        "interpretation_class":"temporal_alignment","evidence_status":"temporally_aligned",
        "bounded_interpretation":"Distribution and promotion actions overlap the FY2024 company-sales period, but the outcome is too broad to isolate either action.",
        "allowed_public_claim":"The actions and company-sales trajectory may be described as temporally aligned.",
        "prohibited_inference":"Do not state that distribution or promotion caused FY2024 sales.",
        "critical_caveat":"Track A portfolio outcomes remain context, not action-level results."
    },
    {
        "case_id":"S6ECASE10","canonical_group":"Unilever Indonesia",
        "case_title":"Small-pack and coinage-pricing volume explanation",
        "strategy_ids":"STR02","action_ids":"S6CACT_UNV_003",
        "result_ids":"","company_claim_ids":"S6CCLM_UNV_001",
        "stage4_context_ids":"FND4_02;FND4_03;FND4_04",
        "interpretation_class":"company_reported_attribution","evidence_status":"company_reported",
        "bounded_interpretation":"Small-pack/coinage pricing is documented and management reported a volume effect, but no compatible OUT09 volume observation was extracted.",
        "allowed_public_claim":"The stated volume effect may be presented only as management attribution.",
        "prohibited_inference":"Do not present the volume effect as independently observed.",
        "critical_caveat":"Brand-penetration language is not consumer reach or market share."
    },
    {
        "case_id":"S6ECASE11","canonical_group":"Unilever Indonesia",
        "case_title":"Wipol and Trika relaunches without brand-level results",
        "strategy_ids":"STR01","action_ids":"S6CACT_UNV_004;S6CACT_UNV_005",
        "result_ids":"","company_claim_ids":"","stage4_context_ids":"FND4_02;FND4_03;FND4_04",
        "interpretation_class":"evidence_boundary","evidence_status":"not_assessable",
        "bounded_interpretation":"The Wipol and Trika relaunches are documented, but no compatible brand-level result is available.",
        "allowed_public_claim":"The relaunches may be reported as documented product actions.",
        "prohibited_inference":"Do not assign company-wide or Track A portfolio outcomes to these brand actions.",
        "critical_caveat":"Brand actions cannot inherit company-level outcomes without matching evidence."
    },
    {
        "case_id":"S6ECASE12","canonical_group":"Unilever Indonesia",
        "case_title":"Sunlight product, promotion, and pricing without compatible outcomes",
        "strategy_ids":"STR01;STR02;STR03",
        "action_ids":"S6CACT_UNV_006;S6CACT_UNV_007;S6CACT_UNV_009",
        "result_ids":"","company_claim_ids":"","stage4_context_ids":"FND4_02;FND4_03;FND4_04",
        "interpretation_class":"evidence_boundary","evidence_status":"not_assessable",
        "bounded_interpretation":"Sunlight product/packaging, promotion, and pricing actions are documented without compatible brand-level product, marketing, or price/mix results.",
        "allowed_public_claim":"These launch-execution actions may be described factually.",
        "prohibited_inference":"Do not infer purchase, sales, leadership, campaign, pricing, or mix effects.",
        "critical_caveat":"Only the separate distribution-availability pair is directly supported."
    },
    {
        "case_id":"S6ECASE13","canonical_group":"Unilever Indonesia",
        "case_title":"Sunlight Q1 2025 distribution availability",
        "strategy_ids":"STR04","action_ids":"S6CACT_UNV_008",
        "result_ids":"S6CRES_UNV_Q1_2025_OUT17","company_claim_ids":"",
        "stage4_context_ids":"FND4_02;FND4_03;FND4_04",
        "interpretation_class":"direct_descriptive_linkage","evidence_status":"directly_supported",
        "bounded_interpretation":"The action and result share Sunlight, Indonesia, Q1 2025, and the same distribution-availability construct; coverage was reported above 70% of direct stores.",
        "allowed_public_claim":"The launch included broad outlet coverage and reported coverage above 70% of direct stores.",
        "prohibited_inference":"Do not infer sales, market share, consumer reach, profitability, or causal commercial effect.",
        "critical_caveat":"This is an interim launch-distribution measure, not a full-year commercial outcome."
    },
    {
        "case_id":"S6ECASE14","canonical_group":"Unilever Indonesia",
        "case_title":"Ice Cream separation and continuing-operation reporting perimeter",
        "strategy_ids":"STR09","action_ids":"S6CACT_UNV_010",
        "result_ids":"S6CRES_UNV_CONT_FY2025_OUT07;S6CRES_UNV_CONT_FY2024_REPRESENTED_OUT07",
        "company_claim_ids":"","stage4_context_ids":"FND4_06",
        "interpretation_class":"reporting_perimeter_context","evidence_status":"context_only",
        "bounded_interpretation":"The separation explains the continuing-operation reporting perimeter and re-presented comparator, not the underlying performance cause.",
        "allowed_public_claim":"Use the separation to explain ownership and comparator scope.",
        "prohibited_inference":"Do not treat the separation as a causal driver of continuing-operation performance.",
        "critical_caveat":"The separation completed within FY2025 and changes the reporting perimeter."
    },
]

covered_actions = set()
covered_claims = set()
case_rows = []

for row in case_specs:
    action_ids = [x for x in row["action_ids"].split(";") if x]
    result_ids = [x for x in row["result_ids"].split(";") if x]
    claim_ids = [x for x in row["company_claim_ids"].split(";") if x]
    finding_ids = [x for x in row["stage4_context_ids"].split(";") if x]

    if set(action_ids) - valid_action_ids:
        raise RuntimeError(f"Unknown action in {row['case_id']}")
    if set(result_ids) - valid_result_ids:
        raise RuntimeError(f"Unknown result in {row['case_id']}")
    if set(claim_ids) - valid_claim_ids:
        raise RuntimeError(f"Unknown company claim in {row['case_id']}")
    if set(finding_ids) - valid_finding_ids:
        raise RuntimeError(f"Unknown Track A finding in {row['case_id']}")

    covered_actions.update(action_ids)
    covered_claims.update(claim_ids)

    output_row = dict(row)
    output_row["result_summary"] = summarize_results(row["result_ids"])
    case_rows.append(output_row)

stage6e_case_interpretations = pd.DataFrame(case_rows)

ordered_columns = [
    "case_id","canonical_group","case_title","strategy_ids","action_ids",
    "result_ids","company_claim_ids","stage4_context_ids",
    "interpretation_class","evidence_status","result_summary",
    "bounded_interpretation","allowed_public_claim",
    "prohibited_inference","critical_caveat",
]
stage6e_case_interpretations = stage6e_case_interpretations[ordered_columns]

if len(stage6e_case_interpretations) != 14:
    raise RuntimeError("Expected 14 bounded cases.")
if covered_actions != valid_action_ids:
    raise RuntimeError("Not all 22 actions are represented in Stage 6E cases.")
if covered_claims != valid_claim_ids:
    raise RuntimeError("Not all six company claims are represented in Stage 6E cases.")

print(f"Bounded cases: {len(stage6e_case_interpretations)}")
print(stage6e_case_interpretations["interpretation_class"].value_counts())


Bounded cases: 14
interpretation_class
evidence_boundary               5
company_reported_attribution    4
temporal_alignment              3
direct_descriptive_linkage      1
reporting_perimeter_context     1
Name: count, dtype: int64


## Group-Level and Research-Question Synthesis

Produce one bounded synthesis for each focal group and answer all 11 pre-specified Stage 5 strategy–result questions.


In [5]:
group_rows = [
    {
        "canonical_group":"Wings Group",
        "track_a_context":"Selected-category leadership and observed stability remain validated Track A strengths.",
        "documented_strategy_evidence":"Eight documented product, price, distribution, and marketing actions.",
        "bounded_strategy_result_interpretation":"All eight action-level screenings remain not assessable because compatible result observations are unavailable.",
        "evidence_strength_profile":"documented_actions_plus_context_only_portfolio_results",
        "disclosure_or_scope_constraint":"Private-company result disclosure and brand-result granularity are materially limited.",
        "what_can_be_concluded":"Wings demonstrates documented commercial activity alongside independent Track A portfolio strengths.",
        "what_cannot_be_concluded":"The evidence does not identify which actions produced leadership, stability, sales, or profitability.",
        "group_synthesis_status":"supported_with_caveat",
    },
    {
        "canonical_group":"Indofood",
        "track_a_context":"Indofood remains the verified strict-control brand-family breadth leader; momentum and persistence differ by observed block/opportunity.",
        "documented_strategy_evidence":"No discrete Indofood/ICBP action was extracted at the Stage 6C standard; three management-attribution claims are retained.",
        "bounded_strategy_result_interpretation":"Observed results can be paired with integrated-model and volume/productivity explanations only as company-reported attribution.",
        "evidence_strength_profile":"company_reported_results_without_matching_discrete_action",
        "disclosure_or_scope_constraint":"Parent scope extends beyond primary packaged FMCG; ICBP includes overseas activity.",
        "what_can_be_concluded":"Track A breadth leadership and reported company results may be stated independently, with management explanations labelled.",
        "what_cannot_be_concluded":"The evidence does not independently establish that the integrated model or productivity caused the results.",
        "group_synthesis_status":"supported_with_caveat",
    },
    {
        "canonical_group":"Mayora",
        "track_a_context":"Ownership sensitivity does not alter Mayora's primary breadth position or the overall-winner conclusion.",
        "documented_strategy_evidence":"Four actions cover innovation, pricing, capital allocation, and domestic-input sourcing.",
        "bounded_strategy_result_interpretation":"Innovation and sourcing have temporal alignment; pricing is company-reported attribution; buyback has no compatible consumer outcome.",
        "evidence_strength_profile":"temporal_alignment_plus_company_reported_attribution",
        "disclosure_or_scope_constraint":"Consolidated results mix domestic/export activity and input costs are a material alternative explanation.",
        "what_can_be_concluded":"Selected actions can be discussed alongside company results with explicit temporal, geography, and company-reporting caveats.",
        "what_cannot_be_concluded":"The evidence does not isolate strategy effects on Indonesian household demand, sales, or operating profit.",
        "group_synthesis_status":"supported_with_caveat",
    },
    {
        "canonical_group":"Unilever Indonesia",
        "track_a_context":"Unilever leads 3 of 5 complete focal-category snapshots and retains all three observed incumbent opportunities; momentum leadership is shared by block.",
        "documented_strategy_evidence":"Ten actions cover distribution, promotion, pricing, relaunches, Sunlight execution, and Ice Cream separation.",
        "bounded_strategy_result_interpretation":"One Sunlight distribution pair is directly supported descriptively; two company-level actions are temporal; one volume explanation is company-reported; one disposal action is context; five actions are not assessable.",
        "evidence_strength_profile":"one_direct_descriptive_pair_plus_temporal_and_company_reported_evidence",
        "disclosure_or_scope_constraint":"Brand-result gaps, interim-measure limits, alternative factors, and reporting-perimeter changes constrain interpretation.",
        "what_can_be_concluded":"The only direct descriptive pair in current screening is Sunlight launch distribution coverage above 70% of direct stores.",
        "what_cannot_be_concluded":"This does not establish superior overall strategy effectiveness or explain Track A leadership, momentum, persistence, sales, or profitability.",
        "group_synthesis_status":"supported_with_caveat",
    },
]

stage6e_group_synthesis = pd.DataFrame(group_rows)

if len(stage6e_group_synthesis) != 4:
    raise RuntimeError("Expected four group-synthesis rows.")

rq_specs = {
    "SRQ01":("supported_with_caveat","Stage 6C documents 22 actions: Wings 8, Mayora 4, Unilever Indonesia 10; Indofood/ICBP instead contributes management-attribution claims at the current extraction standard.","Action counts reflect evidence coverage, not strategy intensity or effectiveness."),
    "SRQ02":("supported_with_caveat","Each documented action retains an intended mechanism, scope, geography, ownership treatment, and period, but intended mechanism is not realized causal effect.","Mechanism description is not impact identification."),
    "SRQ03":("partially_supported","Portfolio/product actions have limited alignment with inherited outcomes: Mayora innovation is temporal; most Wings and Unilever brand actions lack compatible brand results; Track A findings otherwise remain context.","Group-level breadth/leadership/stability/persistence cannot be assigned to brand actions without matching evidence."),
    "SRQ04":("partially_supported","Pricing evidence remains heterogeneous: Wings pack prices lack outcomes; Mayora pricing has company-reported sales attribution; Unilever small-pack pricing has a company-reported volume effect without OUT09; Sunlight NRM lacks price/mix outcome.","No pricing action supports an independently identified causal effect."),
    "SRQ05":("partially_supported","Marketing evidence supports documented activity and limited temporal context. Unilever standardized promotion is temporal with company sales; Wings ProGuard and Sunlight promotion lack compatible brand outcomes.","Temporal association is not marketing-effect identification."),
    "SRQ06":("supported_with_caveat","Sunlight Q1 2025 provides the only direct descriptive pair: broad outlet coverage with reported coverage above 70% of direct stores. Unilever company distribution is temporal with FY2024 sales; Mayora sourcing is temporal with operating profit; Wings distribution lacks a result.","Distribution availability is not consumer reach and sourcing alignment is not causal evidence."),
    "SRQ07":("partially_supported","Mayora buyback has no compatible consumer outcome. ICBP productivity remains management attribution to FY2024 results. No independent capital-allocation or productivity effect is identified.","Capital allocation, productivity explanation, and consumer-market success are distinct constructs."),
    "SRQ08":("supported","Reporting perimeter remains explicit: Indofood parent is broader than packaged FMCG; ICBP and Mayora are mixed geography; Unilever original and re-presented comparators remain separate; Wings has limited public company-result disclosure.","Company/consolidated results cannot be reassigned to brands or Indonesian household demand."),
    "SRQ09":("supported","Alternative explanations materially constrain interpretation, including raw-material costs, mixed geography, ownership/disposal changes, reporting-perimeter changes, missing brand results, and disclosure asymmetry.","These constraints block broad strategy-effect claims."),
    "SRQ10":("not_comparable_for_effect_ranking","The evidence does not support a cross-group strategy-effect ranking because evidence classes and assessability differ with disclosure, scope, geography, and granularity.","Linkage-status counts are evidence diagnostics, not a performance scale."),
    "SRQ11":("supported_no_change","Combined Track A and Track B evidence does not change validated dimension leaders and still does not make one overall winner defensible.","No pre-specified comparable strategy-effect scale or weighting framework exists."),
}

rq_rows = []
for question in research_questions.itertuples(index=False):
    status, answer, limit = rq_specs[question.question_id]
    rq_rows.append({
        "question_id":question.question_id,
        "question_domain":question.question_domain,
        "research_question":question.research_question,
        "synthesis_status":status,
        "bounded_answer":answer,
        "primary_evidence":"stage6e_case_interpretations; stage6e_group_synthesis; stage4_final_findings",
        "critical_limit":limit,
    })

stage6e_research_question_synthesis = pd.DataFrame(rq_rows)

if len(stage6e_research_question_synthesis) != 11:
    raise RuntimeError("Expected 11 research-question synthesis rows.")

print("Group synthesis rows:", len(stage6e_group_synthesis))
print("Research-question rows:", len(stage6e_research_question_synthesis))


Group synthesis rows: 4
Research-question rows: 11


## Cross-Track Synthesis, Findings, and Exceptions

Integrate Track A and Track B without post-hoc reweighting, register bounded findings, and preserve residual interpretation limits.


In [6]:
cross_track_rows = [
    ("S6ECTS01","portfolio_breadth",
     "Indofood leads verified strict-control brand-family breadth at 38, ahead of Wings 37, Unilever 25, and Mayora 22.",
     "Track B contains no comparable action-effect evidence capable of reweighting structural breadth.",
     "Indofood remains the breadth leader on the validated Track A definition.",
     "unchanged",
     "Breadth is not category breadth or strategy effectiveness."),
    ("S6ECTS02","selected_category_leadership",
     "Unilever leads 3 of 5 complete selected focal-category snapshots and Wings leads 2 of 5.",
     "No defensible action-level explanation is established for those group-level leadership observations.",
     "Selected-category leadership remains a Track A outcome rather than a validated result of specific actions.",
     "unchanged",
     "Selected focal-category leadership is not market share."),
    ("S6ECTS03","stability_and_momentum",
     "Wings leads observed stability in 4 of 5 comparable blocks; momentum leadership is split between Indofood and Unilever at 2 blocks each and Wings at 1.",
     "Track B evidence is too sparse and granularly mismatched to explain the block-level pattern.",
     "Track A stability and momentum leaders remain unchanged.",
     "unchanged",
     "Stability is not strength and momentum magnitudes are not averaged across categories."),
    ("S6ECTS04","competitive_persistence",
     "Unilever retains all three observed incumbent opportunities; Indofood loses its one observed opportunity; Wings records one challenger takeover.",
     "No screened action-result chain defensibly explains these role-specific outcomes.",
     "Persistence remains an observed Track A competitive-role outcome.",
     "unchanged",
     "Opportunity counts differ across groups."),
    ("S6ECTS05","strategy_result_evidence",
     "Track A does not score strategy effectiveness.",
     "Track B yields one direct descriptive pair, four temporal action pairs, two company-reported action screenings, one reporting-perimeter context case, and fourteen not-assessable actions.",
     "Strategy-result evidence is case-specific and disclosure-sensitive rather than a cross-group effect scale.",
     "adds_new_dimension_without_reweighting_track_a",
     "Evidence-class counts measure assessability, not success."),
    ("S6ECTS06","overall_winner_defensibility",
     "A single overall portfolio winner is not defensible because validated dimensions have different leaders and no common defensible scale/weighting exists.",
     "Track B adds heterogeneous case-level evidence but no comparable cross-group strategy-effect scale.",
     "Combined evidence continues to support dimension-specific leaders rather than one overall winner.",
     "unchanged",
     "A richer narrative does not justify post-hoc weighting."),
]

stage6e_cross_track_synthesis = pd.DataFrame(
    cross_track_rows,
    columns=[
        "synthesis_id","analytical_dimension","track_a_conclusion",
        "track_b_contribution","integrated_conclusion",
        "overall_effect_on_prior_conclusion","critical_caveat",
    ],
)

finding_rows = [
    ("FND6E_01","Documented strategy coverage is not strategy effectiveness",
     "The evidence documents 22 actions across Wings, Mayora, and Unilever Indonesia, while Indofood/ICBP contributes management-attribution claims rather than matching discrete actions; these differences reflect evidence structure and disclosure, not performance ranking.",
     "strategy_evidence_coverage","supported_with_caveat",
     "data/analytical/stage6e_group_synthesis.csv",
     "Action and claim counts must not be ranked as strategy quality or execution intensity."),
    ("FND6E_02","Direct strategy–result support is rare",
     "Only one screened pair is directly supported descriptively: the Sunlight Q1 2025 launch included broad outlet coverage and the company reported coverage above 70% of direct stores.",
     "direct_descriptive_linkage","supported",
     "data/analytical/stage6e_case_interpretations.csv",
     "This is distribution availability, not consumer reach, market share, sales impact, or causal effectiveness."),
    ("FND6E_03","Wings has documented actions but insufficient compatible public results for linkage",
     "Wings has eight documented product, pricing, distribution, and marketing actions, but all remain not assessable at action-result level because compatible result observations are unavailable.",
     "wings_strategy_result_boundary","supported_with_caveat",
     "data/analytical/stage6e_case_interpretations.csv",
     "This is a disclosure/evidence limitation, not evidence of weak performance."),
    ("FND6E_04","Indofood and ICBP result explanations remain company-reported",
     "Indofood and ICBP report observable sales and profit outcomes and management explanations involving the integrated model, volume, and productivity, but those outcomes cannot be independently attributed to a matching discrete strategy action.",
     "indofood_company_attribution","supported_with_caveat",
     "data/analytical/stage6e_case_interpretations.csv",
     "Parent and ICBP reporting perimeters are broader or more geographically mixed than Indonesian household demand."),
    ("FND6E_05","Mayora supports bounded temporal and company-reported interpretation",
     "Mayora product innovation and domestic-input sourcing are temporally aligned with company-level outcomes, while pricing is supported only through company-reported attribution; raw-material costs and mixed geography constrain interpretation.",
     "mayora_bounded_linkage","supported_with_caveat",
     "data/analytical/stage6e_case_interpretations.csv",
     "The evidence does not isolate effects on Indonesian household demand, sales, or operating profit."),
    ("FND6E_06","Unilever contains the only direct descriptive linkage in the current screening",
     "The Sunlight distribution case is the only direct descriptive action-result pair; other Unilever evidence is temporal, company-reported, reporting-perimeter context, or not assessable.",
     "unilever_bounded_linkage","supported_with_caveat",
     "data/analytical/stage6e_case_interpretations.csv",
     "This does not establish superior overall strategy effectiveness or explain Track A portfolio outcomes."),
    ("FND6E_07","Track B does not overturn Track A leaders or overall-winner defensibility",
     "Combined evidence leaves validated Track A dimension leaders unchanged and still does not support one overall winner because strategy-result evidence is heterogeneous, case-specific, and not measured on a comparable common scale.",
     "cross_track_overall_synthesis","supported",
     "data/analytical/stage6e_cross_track_synthesis.csv",
     "No post-hoc weighting or strategy-effect score is introduced."),
]

stage6e_findings = pd.DataFrame(
    finding_rows,
    columns=[
        "finding_id","finding_title","finding_statement","finding_scope",
        "finding_status","primary_evidence","critical_caveat",
    ],
)

exception_rows = [
    ("S6EEX001","Wings Group","disclosure_asymmetry","caveat","open",
     "Compatible public result observations remain unavailable for all eight documented Wings actions.",
     "Treat as evidence limitation rather than weak strategy or performance."),
    ("S6EEX002","Indofood","discrete_action_gap","caveat","open",
     "Indofood/ICBP management explanations do not map to separately extracted discrete actions.",
     "Retain company_reported treatment."),
    ("S6EEX003","Indofood","reporting_perimeter","caveat","controlled",
     "Indofood parent results extend beyond primary packaged FMCG and ICBP includes overseas activity.",
     "Do not treat consolidated outcomes as Indonesian brand or household-demand outcomes."),
    ("S6EEX004","Mayora","mixed_geography","caveat","controlled",
     "Mayora consolidated outcomes mix domestic and export/international activity.",
     "Do not interpret temporal alignment as Indonesia-only demand effect."),
    ("S6EEX005","Mayora","alternative_input_cost_factor","caveat","controlled",
     "Raw-material cost pressure is a material alternative explanation for FY2025 sales/profit interpretation.",
     "Retain it alongside pricing and sourcing evidence."),
    ("S6EEX006","Unilever Indonesia","brand_result_granularity","caveat","open",
     "Several Wipol, Trika, and Sunlight actions lack compatible brand-level results.",
     "Do not assign company-wide or Track A outcomes to those actions."),
    ("S6EEX007","Unilever Indonesia","interim_distribution_measure","caveat","controlled",
     "The direct Sunlight pair uses a Q1 2025 launch measure rather than a full-year commercial outcome.",
     "Limit interpretation to reported launch distribution availability."),
    ("S6EEX008","Unilever Indonesia","reporting_perimeter_change","caveat","controlled",
     "Ice Cream separation requires a re-presented FY2024 continuing-operation comparator for FY2025 interpretation.",
     "Keep original and re-presented FY2024 observations distinct."),
    ("S6EEX009","Cross-company context","evidence_class_heterogeneity","caveat","controlled",
     "Direct, temporal, company-reported, context-only, and not-assessable evidence are not a common quantitative scale.",
     "Do not rank strategy effectiveness from evidence-status counts."),
    ("S6EEX010","Cross-company context","track_granularity_mismatch","caveat","controlled",
     "Track A portfolio findings generally operate at broader or different granularity than Stage 6C actions.",
     "Use Track A as context unless action/category/period alignment is explicit."),
    ("S6EEX011","Cross-company context","causal_identification_absent","caveat","controlled",
     "No defensible causal-identification design exists for the screened cases.",
     "Use descriptive, temporal, company-reported, or context-only language."),
    ("S6EEX012","Cross-company context","overall_scale_absent","caveat","controlled",
     "No pre-specified comparable strategy-effect scale or weighting framework exists across focal groups.",
     "Keep the overall-winner conclusion unchanged."),
]

stage6e_interpretation_exceptions = pd.DataFrame(
    exception_rows,
    columns=[
        "exception_id","canonical_group","exception_type","severity",
        "status","description","required_treatment",
    ],
)

for df, expected, label in [
    (stage6e_cross_track_synthesis,6,"cross-track rows"),
    (stage6e_findings,7,"findings"),
    (stage6e_interpretation_exceptions,12,"exceptions"),
]:
    if len(df) != expected:
        raise RuntimeError(f"Unexpected {label}: {len(df)}")

print("Cross-track rows:", len(stage6e_cross_track_synthesis))
print("Findings:", len(stage6e_findings))
print("Exceptions:", len(stage6e_interpretation_exceptions))


Cross-track rows: 6
Findings: 7
Exceptions: 12


## Validation, Canonical Outputs, and Manifest

Validate semantic boundaries and completeness, write the eight canonical Stage 6E outputs, create SHA-256 checksums, and run final QA.


In [7]:
validation_specs = [
    ("S6E001","input_integrity","All 11 governed inputs match locked SHA-256 values.","11/11 inputs passed","passed","no"),
    ("S6E002","prior_stage_gate","Stage 6D final gate remains PASS_WITH_CAVEAT.","PASS_WITH_CAVEAT","passed_with_caveat","no"),
    ("S6E003","action_coverage","All 22 Stage 6C actions are represented in bounded cases.","22/22 actions","passed","no"),
    ("S6E004","claim_coverage","All six company-attribution claims are represented in bounded cases.","6/6 claims","passed","no"),
    ("S6E005","case_identity","Fourteen bounded cases are unique.","14/14 unique","passed","no"),
    ("S6E006","group_coverage","All four focal groups have group synthesis.","4/4 groups","passed","no"),
    ("S6E007","research_question_coverage","All 11 pre-specified research questions are answered.","11/11 questions","passed","no"),
    ("S6E008","cross_track_coverage","Six cross-track synthesis dimensions are retained.","6 rows","passed","no"),
    ("S6E009","finding_registry","Seven bounded findings are registered.","7 findings","passed","no"),
    ("S6E010","direct_pair","Sunlight Q1 2025 distribution remains the only direct descriptive pair.","1 direct pair","passed_with_caveat","no"),
    ("S6E011","distribution_semantics","Direct-store coverage remains distribution availability, not consumer reach.","retained","passed","no"),
    ("S6E012","temporal_boundary","Temporal alignment is not upgraded to causation.","retained","passed","no"),
    ("S6E013","company_claim_boundary","Company explanations remain company-reported.","6/6 constrained","passed","no"),
    ("S6E014","context_boundary","Ice Cream separation remains reporting-perimeter context.","retained","passed","no"),
    ("S6E015","missingness","Missing compatible outcomes remain evidence boundaries.","preserved","passed","no"),
    ("S6E016","wings_neutrality","Wings disclosure limitations are not interpreted as weak performance.","retained","passed_with_caveat","no"),
    ("S6E017","indofood_action_gap","Indofood/ICBP explanations remain company-reported without matching discrete actions.","retained","passed_with_caveat","no"),
    ("S6E018","indofood_scope","Indofood parent scope remains broader than packaged FMCG.","retained","passed_with_caveat","no"),
    ("S6E019","icbp_geography","ICBP mixed geography is retained.","retained","passed_with_caveat","no"),
    ("S6E020","mayora_geography","Mayora mixed geography is retained.","retained","passed_with_caveat","no"),
    ("S6E021","mayora_input_costs","Raw-material cost pressure remains an alternative factor.","retained","passed_with_caveat","no"),
    ("S6E022","mayora_buyback","Buyback remains capital-allocation context without consumer outcome.","retained","passed","no"),
    ("S6E023","unilever_brand_granularity","Brand actions without brand results are not assigned company-wide outcomes.","retained","passed","no"),
    ("S6E024","unilever_volume_claim","Small-pack volume effect remains company-reported without OUT09.","retained","passed_with_caveat","no"),
    ("S6E025","unilever_interim_scope","Sunlight direct pair remains interim launch evidence.","retained","passed_with_caveat","no"),
    ("S6E026","unilever_comparator_scope","FY2025 continuing-operation context uses re-presented FY2024 comparator.","retained","passed","no"),
    ("S6E027","track_a_dimensions","Track A breadth, leadership, stability/momentum, persistence, and sensitivity conclusions remain unchanged.","unchanged","passed","no"),
    ("S6E028","overall_winner","A single overall winner remains not defensible.","unchanged","passed","no"),
    ("S6E029","cross_group_ranking","Evidence-status counts are not used as strategy-effect rankings.","none created","passed","no"),
    ("S6E030","causal_boundary","No case or finding upgrades evidence to causal identification.","0 upgrades","passed","no"),
    ("S6E031","market_share_boundary","No non-market-share metric is relabelled as market share.","0 upgrades","passed","no"),
    ("S6E032","composite_boundary","No score, normalization, weighting, or composite is created.","none created","passed","no"),
    ("S6E033","reporting_boundary","No report, README, or visualization is created.","none created","passed","no"),
    ("S6E034","exception_registry","Twelve interpretation exceptions are retained.","12 exceptions","passed_with_caveat","no"),
    ("S6E035","output_completeness","All eight canonical Stage 6E tables are prepared.","8 tables","passed","no"),
    ("S6E036","stage_gate","Bounded interpretation and cross-track synthesis are complete enough for visualization and final findings consolidation.","PASS_WITH_CAVEAT","passed_with_caveat","no"),
]

stage6e_synthesis_validation = pd.DataFrame(
    [
        {
            "check_id":a,
            "validation_area":b,
            "check_description":c,
            "result":d,
            "status":e,
            "critical_failure":f,
            "required_treatment":"Carry the validated evidence boundary forward."
        }
        for a,b,c,d,e,f in validation_specs
    ]
)

if set(stage6e_synthesis_validation["check_id"]) != {
    f"S6E{i:03d}" for i in range(1,37)
}:
    raise RuntimeError("Stage 6E validation registry is incomplete.")

if (stage6e_synthesis_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 6E contains a critical validation failure.")

final_stage6e = stage6e_synthesis_validation.loc[
    stage6e_synthesis_validation["check_id"] == "S6E036"
].iloc[0]

if (
    final_stage6e["result"] != "PASS_WITH_CAVEAT"
    or final_stage6e["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6E final gate.")

output_frames = {
    "metadata/stage6e_input_lock.csv":stage6e_input_lock,
    "data/analytical/stage6e_case_interpretations.csv":stage6e_case_interpretations,
    "data/analytical/stage6e_group_synthesis.csv":stage6e_group_synthesis,
    "data/analytical/stage6e_research_question_synthesis.csv":stage6e_research_question_synthesis,
    "data/analytical/stage6e_cross_track_synthesis.csv":stage6e_cross_track_synthesis,
    "data/analytical/stage6e_findings.csv":stage6e_findings,
    "metadata/stage6e_interpretation_exceptions.csv":stage6e_interpretation_exceptions,
    "metadata/stage6e_synthesis_validation.csv":stage6e_synthesis_validation,
}

for relative_path, dataframe in output_frames.items():
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, encoding="utf-8")

expected_rows = {
    "metadata/stage6e_input_lock.csv":11,
    "data/analytical/stage6e_case_interpretations.csv":14,
    "data/analytical/stage6e_group_synthesis.csv":4,
    "data/analytical/stage6e_research_question_synthesis.csv":11,
    "data/analytical/stage6e_cross_track_synthesis.csv":6,
    "data/analytical/stage6e_findings.csv":7,
    "metadata/stage6e_interpretation_exceptions.csv":12,
    "metadata/stage6e_synthesis_validation.csv":36,
}

manifest_rows = []
for relative_path, dataframe in output_frames.items():
    manifest_rows.append({
        "file_path":relative_path,
        "artifact_type":"csv",
        "row_count":len(dataframe),
        "sha256":sha256_file(OUTPUT_ROOT / relative_path),
        "locked_input_commit":INPUT_COMMIT,
    })

stage6e_output_manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / "metadata/stage6e_output_manifest.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
stage6e_output_manifest.to_csv(manifest_path, index=False, encoding="utf-8")

for relative_path, expected in expected_rows.items():
    actual = len(pd.read_csv(
        OUTPUT_ROOT / relative_path,
        dtype=str, keep_default_na=False
    ))
    if actual != expected:
        raise RuntimeError(
            f"{relative_path}: expected {expected}, found {actual}"
        )

manifest_check = pd.read_csv(
    manifest_path, dtype=str, keep_default_na=False
)

if len(manifest_check) != 8:
    raise RuntimeError("Expected 8 manifest rows.")

for row in manifest_check.itertuples(index=False):
    if sha256_file(OUTPUT_ROOT / row.file_path) != row.sha256:
        raise RuntimeError(f"Manifest hash mismatch: {row.file_path}")

binary_extensions = {
    ".pdf",".doc",".docx",".xls",".xlsx",".ppt",".pptx",
    ".jpg",".jpeg",".png",".webp",".zip"
}
unexpected_binaries = [
    str(path)
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in binary_extensions
]
if unexpected_binaries:
    raise RuntimeError(f"Unexpected binary/raw outputs: {unexpected_binaries}")

overall = stage6e_cross_track_synthesis.loc[
    stage6e_cross_track_synthesis["analytical_dimension"]
    == "overall_winner_defensibility"
].iloc[0]

if overall["overall_effect_on_prior_conclusion"] != "unchanged":
    raise RuntimeError("Overall-winner conclusion changed unexpectedly.")

print("Stage 6E final QA passed.")
print("Locked inputs: 11")
print("Bounded cases: 14")
print("Group synthesis rows: 4")
print("Research-question rows: 11")
print("Cross-track rows: 6")
print("Findings: 7")
print("Exceptions: 12")
print("Validation checks: 36")
print("Critical failures: 0")
print("Causal effects estimated: 0")
print("Strategy-effect rankings created: 0")
print("Composite scores created: 0")
print("Visualizations created: 0")
print("Reports/README created: 0")
print("Raw copyrighted source files written: 0")
print("Manifest hashes: 8/8 matched")
print(f"Stage 6E gate: {final_stage6e['result']} / {final_stage6e['status']}")

display(stage6e_group_synthesis)
display(stage6e_findings)


Stage 6E final QA passed.
Locked inputs: 11
Bounded cases: 14
Group synthesis rows: 4
Research-question rows: 11
Cross-track rows: 6
Findings: 7
Exceptions: 12
Validation checks: 36
Critical failures: 0
Causal effects estimated: 0
Strategy-effect rankings created: 0
Composite scores created: 0
Visualizations created: 0
Reports/README created: 0
Raw copyrighted source files written: 0
Manifest hashes: 8/8 matched
Stage 6E gate: PASS_WITH_CAVEAT / passed_with_caveat


,canonical_group,track_a_context,documented_strategy_evidence,bounded_strategy_result_interpretation,evidence_strength_profile,disclosure_or_scope_constraint,what_can_be_concluded,what_cannot_be_concluded,group_synthesis_status
0,Wings Group,Selected-category leadership and observed stability remain validated Track A strengths.,"Eight documented product, price, distribution, and marketing actions.",All eight action-level screenings remain not assessable because compatible result observations are unavailable.,documented_actions_plus_context_only_portfolio_results,Private-company result disclosure and brand-result granularity are materially limited.,Wings demonstrates documented commercial activity alongside independent Track A portfolio strengths.,"The evidence does not identify which actions produced leadership, stability, sales, or profitability.",supported_with_caveat
1,Indofood,Indofood remains the verified strict-control brand-family breadth leader; momentum and persistence differ by observed block/opportunity.,No discrete Indofood/ICBP action was extracted at the Stage 6C standard; three management-attribution claims are retained.,Observed results can be paired with integrated-model and volume/productivity explanations only as company-reported attribution.,company_reported_results_without_matching_discrete_action,Parent scope extends beyond primary packaged FMCG; ICBP includes overseas activity.,"Track A breadth leadership and reported company results may be stated independently, with management explanations labelled.",The evidence does not independently establish that the integrated model or productivity caused the results.,supported_with_caveat
2,Mayora,Ownership sensitivity does not alter Mayora's primary breadth position or the overall-winner conclusion.,"Four actions cover innovation, pricing, capital allocation, and domestic-input sourcing.",Innovation and sourcing have temporal alignment; pricing is company-reported attribution; buyback has no compatible consumer outcome.,temporal_alignment_plus_company_reported_attribution,Consolidated results mix domestic/export activity and input costs are a material alternative explanation.,"Selected actions can be discussed alongside company results with explicit temporal, geography, and company-reporting caveats.","The evidence does not isolate strategy effects on Indonesian household demand, sales, or operating profit.",supported_with_caveat
3,Unilever Indonesia,Unilever leads 3 of 5 complete focal-category snapshots and retains all three observed incumbent opportunities; momentum leadership is shared by block.,"Ten actions cover distribution, promotion, pricing, relaunches, Sunlight execution, and Ice Cream separation.",One Sunlight distribution pair is directly supported descriptively; two company-level actions are temporal; one volume explanation is company-reported; one disposal action is c...,one_direct_descriptive_pair_plus_temporal_and_company_reported_evidence,"Brand-result gaps, interim-measure limits, alternative factors, and reporting-perimeter changes constrain interpretation.",The only direct descriptive pair in current screening is Sunlight launch distribution coverage above 70% of direct stores.,"This does not establish superior overall strategy effectiveness or explain Track A leadership, momentum, persistence, sales, or profitability.",supported_with_caveat


,finding_id,finding_title,finding_statement,finding_scope,finding_status,primary_evidence,critical_caveat
0,FND6E_01,Documented strategy coverage is not strategy effectiveness,"The evidence documents 22 actions across Wings, Mayora, and Unilever Indonesia, while Indofood/ICBP contributes management-attribution claims rather than matching discrete acti...",strategy_evidence_coverage,supported_with_caveat,data/analytical/stage6e_group_synthesis.csv,Action and claim counts must not be ranked as strategy quality or execution intensity.
1,FND6E_02,Direct strategy–result support is rare,Only one screened pair is directly supported descriptively: the Sunlight Q1 2025 launch included broad outlet coverage and the company reported coverage above 70% of direct sto...,direct_descriptive_linkage,supported,data/analytical/stage6e_case_interpretations.csv,"This is distribution availability, not consumer reach, market share, sales impact, or causal effectiveness."
2,FND6E_03,Wings has documented actions but insufficient compatible public results for linkage,"Wings has eight documented product, pricing, distribution, and marketing actions, but all remain not assessable at action-result level because compatible result observations ar...",wings_strategy_result_boundary,supported_with_caveat,data/analytical/stage6e_case_interpretations.csv,"This is a disclosure/evidence limitation, not evidence of weak performance."
3,FND6E_04,Indofood and ICBP result explanations remain company-reported,"Indofood and ICBP report observable sales and profit outcomes and management explanations involving the integrated model, volume, and productivity, but those outcomes cannot be...",indofood_company_attribution,supported_with_caveat,data/analytical/stage6e_case_interpretations.csv,Parent and ICBP reporting perimeters are broader or more geographically mixed than Indonesian household demand.
4,FND6E_05,Mayora supports bounded temporal and company-reported interpretation,"Mayora product innovation and domestic-input sourcing are temporally aligned with company-level outcomes, while pricing is supported only through company-reported attribution; ...",mayora_bounded_linkage,supported_with_caveat,data/analytical/stage6e_case_interpretations.csv,"The evidence does not isolate effects on Indonesian household demand, sales, or operating profit."
5,FND6E_06,Unilever contains the only direct descriptive linkage in the current screening,"The Sunlight distribution case is the only direct descriptive action-result pair; other Unilever evidence is temporal, company-reported, reporting-perimeter context, or not ass...",unilever_bounded_linkage,supported_with_caveat,data/analytical/stage6e_case_interpretations.csv,This does not establish superior overall strategy effectiveness or explain Track A portfolio outcomes.
6,FND6E_07,Track B does not overturn Track A leaders or overall-winner defensibility,"Combined evidence leaves validated Track A dimension leaders unchanged and still does not support one overall winner because strategy-result evidence is heterogeneous, case-spe...",cross_track_overall_synthesis,supported,data/analytical/stage6e_cross_track_synthesis.csv,No post-hoc weighting or strategy-effect score is introduced.
